## **Images.txt Generation**

In [21]:
from google.colab import drive
import os
import re
import pandas as pd

In [22]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
base_dir = '/content/drive/MyDrive/Bird_Species_Detection/manually_selected_images/'

In [24]:
outputBaseDir = '/content/drive/MyDrive/Bird_Species_Detection/files/Manually_Added_Images_Processing'

In [25]:
output_file = f'{outputBaseDir}/modifiedImages.txt'

In [26]:
counter = 11788

with open(output_file, 'w') as f:
    for species_folder in sorted(os.listdir(base_dir)):
        species_path = os.path.join(base_dir, species_folder)

        if os.path.isdir(species_path):
            for image_name in sorted(os.listdir(species_path)):
                # Skip non-image files
                if not image_name.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    continue

                relative_path = f"{species_folder}/{image_name}"
                f.write(f"{counter} {relative_path}\n")
                counter += 1

print(f"✅ Modified metadata file saved successfully at:\n{output_file}")

✅ Modified metadata file saved successfully at:
/content/drive/MyDrive/Bird_Species_Detection/files/Manually_Added_Images_Processing/modifiedImages.txt


## **image_class_labels.txt Generation**

In [27]:
output_file = f'{outputBaseDir}/modifiedClassLabels.txt'

In [28]:
image_id = 11788
with open(output_file, 'w') as f:
    for species_folder in sorted(os.listdir(base_dir)):
        species_path = os.path.join(base_dir, species_folder)

        if os.path.isdir(species_path):
            # Extract numeric prefix before the first '.'
            match = re.match(r"^(\d+)\.", species_folder)
            if match:
                class_label = int(match.group(1))
            else:
                # If folder doesn't have a numeric prefix, skip it
                print(f"⚠️ Skipping folder without numeric prefix: {species_folder}")
                continue

            # List all image files
            image_files = sorted([
                img for img in os.listdir(species_path)
                if img.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
            ])

            # Write lines: "<image_id> <class_label>"
            for _ in image_files:
                f.write(f"{image_id} {class_label}\n")
                image_id += 1

print(f"✅ Modified class labels file created at:\n{output_file}")
print(f"📄 Total images labeled: {image_id - 11788}")

✅ Modified class labels file created at:
/content/drive/MyDrive/Bird_Species_Detection/files/Manually_Added_Images_Processing/modifiedClassLabels.txt
📄 Total images labeled: 716


## **Train-Test Split**

In [29]:
images_file = f'{outputBaseDir}/modifiedImages.txt'
output_file = f'{outputBaseDir}/modifiedTrainTestSplit.txt'

# 4️⃣ Generate the file with all split values = 1
with open(images_file, 'r') as infile, open(output_file, 'w') as outfile:
    for line in infile:
        # Get image ID (first number)
        image_id = line.strip().split()[0]
        outfile.write(f"{image_id} 1\n")

print(f"✅ Modified train-test split file created successfully at:\n{output_file}")

✅ Modified train-test split file created successfully at:
/content/drive/MyDrive/Bird_Species_Detection/files/Manually_Added_Images_Processing/modifiedTrainTestSplit.txt


In [30]:
summary_data = []

# Iterate through species folders
for species_folder in sorted(os.listdir(base_dir)):
    species_path = os.path.join(base_dir, species_folder)

    if os.path.isdir(species_path):
        counts = {'.jpg': 0, '.jpeg': 0, '.png': 0, '.webp': 0, 'other': 0}

        for image_name in os.listdir(species_path):
            ext = os.path.splitext(image_name.lower())[1]
            if ext in counts:
                counts[ext] += 1
            else:
                counts['other'] += 1

        total = sum(counts.values())
        summary_data.append([
            species_folder,
            counts['.jpg'],
            counts['.jpeg'],
            counts['.png'],
            counts['.webp'],
            counts['other'],
            total
        ])

# Create DataFrame
summary_df = pd.DataFrame(summary_data, columns=[
    'Species Name', '.jpg', '.jpeg', '.png', '.webp', 'Other Extensions', 'Total Images'
])

# Save to Excel and CSV
output_summary_excel = f'{outputBaseDir}/image_extension_summary.xlsx'
summary_df.to_excel(output_summary_excel, index=False)

print(f"✅ Image extension summary saved as:\n📊 {output_summary_excel}")

✅ Image extension summary saved as:
📊 /content/drive/MyDrive/Bird_Species_Detection/files/Manually_Added_Images_Processing/image_extension_summary.xlsx
